# Lightweight HBCC — CIFAR-100 Full Training Pipeline

This notebook mirrors the CIFAR-10 Kaggle pipeline in `hbcc-cifar-10.ipynb` and applies it to CIFAR-100 by reusing the same configs with overrides:

- `data.name=cifar100`
- `model.num_classes=100`
- CIFAR-100-specific `experiment.name` values and output directories

The training split is `45k train / 5k val` from the original CIFAR-100 training split. The official 10k test split is evaluated once on `best.pth` and reported as `test_acc1` and `test_acc5`.

Scope matches the executed CIFAR-10 run to stay safely inside Kaggle's 12h GPU session cap: teacher, lightweight baselines, CoC baseline, HBCC-Accuracy Small/Medium, pruning export, benchmarks, and the published results table. Proxy search, ablations, and KD stay disabled by default, same as the CIFAR-10 run.


In [1]:
!git clone https://github.com/nvhungus/Lightweight-Context-Cluster.git
%cd Lightweight-Context-Cluster


Cloning into 'HBCC'...
remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (351/351), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 351 (delta 232), reused 340 (delta 228), pack-reused 0 (from 0)
Receiving objects: 100% (351/351), 8.17 MiB | 14.53 MiB/s, done.
Resolving deltas: 100% (232/232), done.
/kaggle/working/HBCC


In [2]:
from __future__ import annotations

import json
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Run this notebook from the repo root or notebooks dir. Got {ROOT}"

COC_PYTHON = Path(r"D:\Anaconda\envs\CoC\python.exe")
PYTHON = COC_PYTHON if COC_PYTHON.exists() else Path(sys.executable)

print("Repo:", ROOT)
print("Python:", PYTHON)

Repo: /kaggle/working/HBCC
Python: /usr/bin/python3


## Phase Switches

Use the same execution style as the CIFAR-10 notebook: start with checks and smoke, then enable one research phase at a time.

In [ ]:
RUN_ENV_CHECKS = True
RUN_SMOKE = False

# Architecture fix: stem_stride=1, depths=[1,1,2,1] — requires three Kaggle sessions for CIFAR-100.
#   Session A — Baselines (current): RUN_LIGHTWEIGHT_BASELINES=True, RUN_COC_BASELINE=True, rest False
#   Session B — CE base (after A):   RUN_STUDENT_CE=True, rest False
#   Session C — KD (after B):        RUN_KD=True, rest False
RUN_TEACHER = True
RUN_LIGHTWEIGHT_BASELINES = True   # need fair CIFAR-100 numbers for MobileNet + ShuffleNet
RUN_COC_BASELINE = True            # retrain CoC baseline for CIFAR-100
RUN_STUDENT_CE = False
RUN_PROXY_SEARCH = False
RUN_ABLATIONS = False
RUN_KD = False
RUN_PRUNING = False
RUN_BENCHMARKS = True
RUN_SUMMARY_REPORT = True
RUN_PARETO_REPORT = True

# Epoch budgets — fixed HBCC architecture (~2× compute per epoch vs old stem_stride=2):
# Session A - Baselines: Teacher(300ep,2.67h) + MobileNet(300ep,~0.7h) + ShuffleNet(300ep,~0.7h) + CoC(300ep,~1.5h) + Bench(0.78h) ≈ 6.4h
# Session B - CE:        Teacher(300ep,2.67h) + CE-Medium(250ep,~4.0h) + CE-Small(250ep,~3.7h) + Bench(0.78h) ≈ 11.1h
# Session C - KD:        Teacher(300ep,2.67h) + KD-Tiny(250ep,~0.75h) + KD-Medium(250ep,~4.0h) + KD-Small(250ep,~3.7h) + Bench(0.78h) ≈ 11.9h
# 250ep × 2× compute ≈ same total gradient work as old 300ep × 1× compute.
FULL_EPOCHS = 300
ACCURACY_EPOCHS = 250   # reduced from 300 to fit 12h with 2× per-epoch compute
KD_MEDIUM_EPOCHS = 250  # same rationale
KD_SMALL_EPOCHS = 250   # same rationale
PROXY_EPOCHS = 30
PRUNING_EPOCHS = 80
PRUNED_FINETUNE_EPOCHS = 80

CIFAR100_OVERRIDES = [
    "data.name=cifar100",
    "model.num_classes=100",
]

COMMON_TRAIN_OVERRIDES = [
    # Reduce these if you hit OOM.
    # "data.batch_size=64",
    # "data.val_batch_size=128",
    # "data.test_batch_size=128",
]

BENCHMARK_OUTPUT = "results/benchmark_cifar100"
BENCHMARK_BATCHES = [1, 16, 64, 128]
BENCHMARK_WARMUP = 30
BENCHMARK_RUNS = 100

DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP = 2
DEBUG_BENCHMARK_RUNS = 3


In [4]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def cifar100_overrides(experiment_name: str | None = None, overrides: list[str] | None = None) -> list[str]:
    items = [*CIFAR100_OVERRIDES]
    if experiment_name is not None:
        items.append(f"experiment.name={experiment_name}")
    items.extend(overrides or [])
    return items


def train(
    config: str,
    output: str,
    experiment_name: str,
    overrides: list[str] | None = None,
    extra: list[str] | None = None,
) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args(cifar100_overrides(experiment_name, [*(overrides or []), *COMMON_TRAIN_OVERRIDES]))
    args += extra or []
    run_py(args)


def benchmark(
    config: str,
    experiment_name: str,
    checkpoint: str | None = None,
    debug: bool = False,
    profile: bool = True,
) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup = DEBUG_BENCHMARK_WARMUP if debug else BENCHMARK_WARMUP
    runs = DEBUG_BENCHMARK_RUNS if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py",
        "--config",
        config,
        "--output",
        BENCHMARK_OUTPUT,
        "--batch-sizes",
        *[str(v) for v in batches],
        "--warmup",
        str(warmup),
        "--runs",
        str(runs),
    ]
    args += override_args(cifar100_overrides(experiment_name))
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"Skip checkpoint for {config}; not found: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)

## 0. Environment And Smoke Checks

In [5]:
if RUN_ENV_CHECKS:
    run_py(["-m", "pytest"])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_latency_tiny.yaml", "--batch-size", "1", *override_args(CIFAR100_OVERRIDES)])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_current_reference.yaml", "--batch-size", "1", *override_args(CIFAR100_OVERRIDES)])

if RUN_SMOKE:
    run_py([
        "tools/train.py",
        "--config",
        "configs/smoke.yaml",
        "--output",
        "runs_smoke_cifar100",
        "--override",
        "experiment.name=smoke_hbcc_latency_tiny_fake_cifar100",
        "--override",
        "model.num_classes=100",
        "--limit-train-batches",
        "1",
        "--limit-val-batches",
        "1",
        "--limit-test-batches",
        "1",
    ])
    benchmark("configs/smoke.yaml", "smoke_hbcc_latency_tiny_fake_cifar100", debug=True, profile=False)


$ /usr/bin/python3 -m pytest


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working/HBCC
configfile: pyproject.toml
testpaths: tests
plugins: langsmith-0.7.30, anyio-4.13.0, typeguard-4.5.1
collected 10 items

tests/test_config_and_pruning.py ..                                      [ 20%]
tests/test_data.py ....                                                  [ 60%]
tests/test_models.py ....                                                [100%]

============================= 10 passed in 11.75s ==============================

[exit=0] elapsed=15.7s

$ /usr/bin/python3 tools/shape_trace.py --config configs/hbcc_latency_tiny.yaml --batch-size 1 --override data.name=cifar100 --override model.num_classes=100


stem                                             -> (1, 32, 16, 16)
stages.0                                         -> (1, 32, 16, 16)
downsamples.0                                    -> (1, 48, 8, 8)
stages.1.blocks.0.cluster                        -> (1, 24, 8, 8)
stages.1                                         -> (1, 48, 8, 8)
downsamples.1                                    -> (1, 96, 4, 4)
stages.2.blocks.0.cluster                        -> (1, 96, 4, 4)
stages.2.blocks.1.cluster                        -> (1, 96, 4, 4)
stages.2                                         -> (1, 96, 4, 4)
downsamples.2                                    -> (1, 160, 2, 2)
stages.3.blocks.0.cluster                        -> (1, 160, 2, 2)
stages.3                                         -> (1, 160, 2, 2)
logits                                           -> (1, 100)

[exit=0] elapsed=5.3s

$ /usr/bin/python3 tools/shape_trace.py --config configs/hbcc_current_reference.yaml --batch-size 1 --override data.

stem                                             -> (1, 32, 32, 32)
stages.0.blocks.0.cluster                        -> (1, 32, 32, 32)
stages.0                                         -> (1, 32, 32, 32)
downsamples.0                                    -> (1, 64, 16, 16)
stages.1.blocks.0.cluster                        -> (1, 32, 16, 16)
stages.1                                         -> (1, 64, 16, 16)
downsamples.1                                    -> (1, 128, 8, 8)
stages.2.blocks.0.cluster                        -> (1, 128, 8, 8)
stages.2.blocks.1.cluster                        -> (1, 128, 8, 8)
stages.2                                         -> (1, 128, 8, 8)
downsamples.2                                    -> (1, 192, 4, 4)
stages.3.blocks.0.cluster                        -> (1, 192, 4, 4)
stages.3                                         -> (1, 192, 4, 4)
logits                                           -> (1, 100)

[exit=0] elapsed=4.5s


## 1. Teacher And Reference Baselines

In [6]:
if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        "runs_teacher_cifar100",
        "resnet18_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        "runs_baselines_cifar100",
        "mobilenet_v2_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        "runs_baselines_cifar100",
        "shufflenet_v2_x1_0_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )


$ /usr/bin/python3 tools/train.py --config configs/baselines/resnet18_cifar.yaml --output runs_teacher_cifar100 --override data.name=cifar100 --override model.num_classes=100 --override experiment.name=resnet18_cifar100 --override train.epochs=200


```text
test:  72%|████████▋   | 29/40 [00:01<00:00, 28.53it/s]
```

setup: dataset=cifar100 output=runs_teacher_cifar100/resnet18_cifar100 device=cuda
setup: building data loaders...

setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2614.6s
setup: building model...
setup: model ready params=11220132 elapsed=2615.0s
train: starting epochs=200 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 3.8661550653626096,
  "train_acc1": 13.428151709401709,
  "train_time_s": 16.7995100359999,
  "val_loss": 3.285864572143555,
  "val_acc1": 19.88,
  "val_time_s": 0.8127284499996676,
  "val_acc5": 48.359999926757816
}

                                                                                     

                                                      
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 3.244396664138

```text
test:  75%|█████████   | 30/40 [00:01<00:00, 30.50it/s]
```

setup: dataset=cifar100 output=runs_baselines_cifar100/mobilenet_v2_cifar100 device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.8s
setup: building model...
setup: model ready params=2351972 elapsed=3.1s
train: starting epochs=200 skip_test=False

                                                                                    

                                                      
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 4.219334357144826,
  "train_acc1": 6.597222222222222,
  "train_time_s": 18.621306285999708,
  "val_loss": 3.741049557876587,
  "val_acc1": 12.120000001525879,
  "val_time_s": 3.1123439449993384,
  "val_acc5": 34.35999996948242
}

                                                                                     

                                                      
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 3.813

```text
test:  72%|████████▋   | 29/40 [00:01<00:00, 28.23it/s]
```

setup: dataset=cifar100 output=runs_baselines_cifar100/shufflenet_v2_x1_0_cifar100 device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.9s
setup: building model...
setup: model ready params=1356104 elapsed=3.1s
train: starting epochs=200 skip_test=False

                                                                                    

                                                      
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 4.167549316699688,
  "train_acc1": 7.507567663817664,
  "train_time_s": 18.27915349399882,
  "val_loss": 3.6624907081604006,
  "val_acc1": 13.239999981689452,
  "val_time_s": 2.318418509001276,
  "val_acc5": 37.31999995727539
}

                                                                                     

                                                      
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 

## 2. CoC Baseline

In [7]:
COC_BASELINE_RUN = "runs_coc_baseline_cifar100/coc_cifar_baseline_cifar100"
COC_BASELINE_CHECKPOINT = f"{COC_BASELINE_RUN}/best.pth"

if RUN_COC_BASELINE:
    train(
        "configs/coc_cifar_baseline.yaml",
        "runs_coc_baseline_cifar100",
        "coc_cifar_baseline_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    run_py(["tools/summarize_training.py", COC_BASELINE_RUN])


$ /usr/bin/python3 tools/train.py --config configs/coc_cifar_baseline.yaml --output runs_coc_baseline_cifar100 --override data.name=cifar100 --override model.num_classes=100 --override experiment.name=coc_cifar_baseline_cifar100 --override train.epochs=200


```text
test:  95%|███████████▍| 38/40 [00:01<00:00, 26.30it/s]
```

setup: dataset=cifar100 output=runs_coc_baseline_cifar100/coc_cifar_baseline_cifar100 device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.8s
setup: building model...
setup: model ready params=2424704 elapsed=3.1s
train: starting epochs=200 skip_test=False

                                                                                     

                                                     
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 3.963605537713423,
  "train_acc1": 11.631944444444445,
  "train_time_s": 20.29797945700011,
  "val_loss": 3.33418632850647,
  "val_acc1": 19.759999990844726,
  "val_time_s": 1.0646672940001736,
  "val_acc5": 47.65999997558594
}

                                                                                     

                                                     
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss"

Best val: epoch=191, val_acc1=60.82, val_loss=1.8433 val_acc5=80.40
Held-out test: checkpoint=best.pth, epoch=191, test_acc1=61.19, test_loss=1.8196 test_acc5=80.66

[exit=0] elapsed=0.1s


## 3. Student CE-Only Training

In [8]:
if RUN_STUDENT_CE:
    train(
        "configs/hbcc_accuracy_small.yaml",
        "runs_accuracy_cifar100",
        "hbcc_accuracy_small_cifar100",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        "runs_accuracy_cifar100",
        "hbcc_accuracy_medium_cifar100",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )



$ /usr/bin/python3 tools/train.py --config configs/hbcc_accuracy_small.yaml --output runs_accuracy_cifar100 --override data.name=cifar100 --override model.num_classes=100 --override experiment.name=hbcc_accuracy_small_cifar100 --override train.epochs=300


```text
test:  92%|███████████ | 37/40 [00:01<00:00, 24.95it/s]
```

setup: dataset=cifar100 output=runs_accuracy_cifar100/hbcc_accuracy_small_cifar100 device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.8s
setup: building model...
setup: model ready params=1558868 elapsed=3.1s
train: starting epochs=300 skip_test=False

                                                                                    

                                                      
{
  "epoch": 0,
  "lr": 0.000208,
  "train_loss": 4.593708694490612,
  "train_acc1": 1.6270477207977208,
  "train_time_s": 28.31879705600295,
  "val_loss": 4.403335838317871,
  "val_acc1": 4.779999998474121,
  "val_time_s": 1.6693270650030172,
  "val_acc5": 17.120000001525877
}

                                                                                    

                                                      
{
  "epoch": 1,
  "lr": 0.00040599999999999995,
  "train_loss": 4.3456804962

```text
test:  82%|█████████▉  | 33/40 [00:01<00:00, 21.11it/s]
```

setup: dataset=cifar100 output=runs_accuracy_cifar100/hbcc_accuracy_medium_cifar100 device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.9s
setup: building model...
setup: model ready params=2863992 elapsed=3.2s
train: starting epochs=300 skip_test=False

                                                                                    

                                                      
{
  "epoch": 0,
  "lr": 0.000208,
  "train_loss": 4.586241193646379,
  "train_acc1": 1.7583689458689458,
  "train_time_s": 30.377894219000154,
  "val_loss": 4.366436323547363,
  "val_acc1": 4.77999999923706,
  "val_time_s": 1.7974442610029655,
  "val_acc5": 17.69999999847412
}

                                                                                    

                                                     
{
  "epoch": 1,
  "lr": 0.00040599999999999995,
  "train_loss": 4.33250947827

## 4. Short Proxy Architecture Search

In [9]:
PROXY_JOBS = [
    {
        "name": "proxy_dims24_depth1111_mlp2_cifar100",
        "overrides": [
            "model.embed_dims=[24,48,96,160]",
            "model.depths=[1,1,1,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_depth1121_mlp2_cifar100",
        "overrides": [
            "model.embed_dims=[32,48,96,160]",
            "model.depths=[1,1,2,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_64_depth1221_mlp3_cifar100",
        "overrides": [
            "model.embed_dims=[32,64,128,192]",
            "model.depths=[1,2,2,1]",
            "model.mlp_ratios=3.0",
            "model.heads=[1,2,4,4]",
        ],
    },
]

if RUN_PROXY_SEARCH:
    for job in PROXY_JOBS:
        train(
            "configs/hbcc_latency_tiny.yaml",
            "runs_proxy_cifar100",
            job["name"],
            overrides=[
                f"train.epochs={PROXY_EPOCHS}",
                *job["overrides"],
            ],
        )

## 5. Independent Ablations

In [10]:
ABLATION_CONFIGS = [
    ("configs/ablations/hbcc_tiny_hamming_late.yaml", "hbcc_tiny_hamming_late_cifar100"),
    ("configs/ablations/hbcc_tiny_lbpconv.yaml", "hbcc_tiny_lbpconv_cifar100"),
]

if RUN_ABLATIONS:
    for cfg, experiment_name in ABLATION_CONFIGS:
        train(cfg, "runs_ablations_cifar100", experiment_name, overrides=[f"train.epochs={FULL_EPOCHS}"])

## 6. Knowledge Distillation

In [ ]:
TEACHER_CHECKPOINT = "runs_teacher_cifar100/resnet18_cifar100/best.pth"

# Timing estimate (Session C - KD): Teacher(2.67h) + KD-Tiny(250ep,~0.75h) + KD-Medium(250ep,~4.0h) + KD-Small(250ep,~3.7h) + Benchmarks(0.78h) ≈ 11.9h
if RUN_KD:
    if not (ROOT / TEACHER_CHECKPOINT).exists():
        raise FileNotFoundError(f"Train the teacher first: {TEACHER_CHECKPOINT}")
    KD_EXTRA = [
        "--teacher-config", "configs/baselines/resnet18_cifar.yaml",
        "--teacher-checkpoint", TEACHER_CHECKPOINT,
    ]
    KD_BASE = ["train.kd_alpha=0.5", "train.kd_temperature=4.0"]

    # Latency-Tiny KD (~0.75h at 250 epochs)
    train(
        "configs/hbcc_latency_tiny.yaml",
        "runs_kd_cifar100",
        "hbcc_latency_tiny_kd_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}", *KD_BASE],
        extra=KD_EXTRA,
    )

    # Accuracy-Medium KD (~4.0h at 250 epochs) — primary CIFAR-100 uplift target
    train(
        "configs/hbcc_accuracy_medium.yaml",
        "runs_kd_cifar100",
        "hbcc_accuracy_medium_kd_cifar100",
        overrides=[f"train.epochs={KD_MEDIUM_EPOCHS}", *KD_BASE],
        extra=KD_EXTRA,
    )

    # Accuracy-Small KD (~3.7h at 250 epochs)
    train(
        "configs/hbcc_accuracy_small.yaml",
        "runs_kd_cifar100",
        "hbcc_accuracy_small_kd_cifar100",
        overrides=[f"train.epochs={KD_SMALL_EPOCHS}", *KD_BASE],
        extra=KD_EXTRA,
    )


## 7. Structured Pruning Export

In [12]:
PRUNING_BASE_CHECKPOINT = "runs_accuracy_cifar100/hbcc_accuracy_small_cifar100/best.pth"
PRUNING_CONFIG = "configs/ablations/hbcc_accuracy_small_pruning_mask.yaml"
PRUNING_RUN_DIR = "runs_pruning_accuracy_cifar100"
PRUNING_EXPERIMENT = "hbcc_accuracy_small_pruning_mask_cifar100"
PRUNING_CHECKPOINT = f"{PRUNING_RUN_DIR}/{PRUNING_EXPERIMENT}/best.pth"
PRUNING_THRESHOLD = "0.90"
PRUNED_CONFIG = "configs/generated_ablations/hbcc_accuracy_small_cifar100_pruned_export.yaml"
PRUNED_FINETUNE_RUN_DIR = "runs_pruned_accuracy_cifar100"
PRUNED_FINETUNE_EXPERIMENT = "hbcc_accuracy_small_cifar100_pruned_export"
PRUNED_FINETUNE_CHECKPOINT = f"{PRUNED_FINETUNE_RUN_DIR}/{PRUNED_FINETUNE_EXPERIMENT}/best.pth"

if RUN_PRUNING:
    if not (ROOT / PRUNING_BASE_CHECKPOINT).exists():
        raise FileNotFoundError(f"Train hbcc_accuracy_small first: {PRUNING_BASE_CHECKPOINT}")
    train(
        PRUNING_CONFIG,
        PRUNING_RUN_DIR,
        PRUNING_EXPERIMENT,
        overrides=[f"train.epochs={PRUNING_EPOCHS}"],
        extra=["--resume", PRUNING_BASE_CHECKPOINT],
    )
    run_py([
        "tools/inspect_pruning_masks.py",
        "--config",
        PRUNING_CONFIG,
        "--checkpoint",
        PRUNING_CHECKPOINT,
        "--thresholds",
        "0.5",
        "0.7",
        "0.8",
        "0.9",
        "0.95",
        *override_args(cifar100_overrides(PRUNING_EXPERIMENT)),
    ])
    run_py([
        "tools/export_pruned.py",
        "--config",
        PRUNING_CONFIG,
        "--checkpoint",
        PRUNING_CHECKPOINT,
        "--output",
        PRUNED_CONFIG,
        "--threshold",
        PRUNING_THRESHOLD,
        "--min-channels",
        "16",
        "--round-to",
        "8",
        *override_args(cifar100_overrides(PRUNED_FINETUNE_EXPERIMENT)),
    ])
    train(
        PRUNED_CONFIG,
        PRUNED_FINETUNE_RUN_DIR,
        PRUNED_FINETUNE_EXPERIMENT,
        overrides=[f"train.epochs={PRUNED_FINETUNE_EPOCHS}"],
    )


$ /usr/bin/python3 tools/train.py --config configs/ablations/hbcc_accuracy_small_pruning_mask.yaml --output runs_pruning_accuracy_cifar100 --override data.name=cifar100 --override model.num_classes=100 --override experiment.name=hbcc_accuracy_small_pruning_mask_cifar100 --override train.epochs=80 --resume runs_accuracy_cifar100/hbcc_accuracy_small_cifar100/best.pth


```text
test:  90%|██████████▊ | 36/40 [00:01<00:00, 24.25it/s]
```

setup: dataset=cifar100 output=runs_pruning_accuracy_cifar100/hbcc_accuracy_small_pruning_mask_cifar100 device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.9s
setup: building model...
setup: model ready params=1559380 elapsed=3.2s
setup: loading checkpoint runs_accuracy_cifar100/hbcc_accuracy_small_cifar100/best.pth
train: starting epochs=80 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.0001999229036240723,
  "train_loss": 1.2262903743999296,
  "train_acc1": 88.55279558404558,
  "train_time_s": 29.151745049006422,
  "val_loss": 1.09963101272583,
  "val_acc1": 72.59999998779297,
  "val_time_s": 1.6679139990010299,
  "val_acc5": 91.5400000366211
}

                                                                                     

           

stage	channels	min	max	mean	keep@0.5	rounded_keep@0.5	keep@0.7	rounded_keep@0.7	keep@0.8	rounded_keep@0.8	keep@0.9	rounded_keep@0.9	keep@0.95	rounded_keep@0.95
0	48	0.9692	0.9705	0.9700	48	48	48	48	48	48	48	48	48	48
1	80	0.9685	0.9706	0.9700	80	80	80	80	80	80	80	80	80	80
2	160	0.9689	0.9724	0.9700	160	160	160	160	160	160	160	160	160	160
3	224	0.9684	0.9748	0.9731	224	224	224	224	224	224	224	224	224	224

[exit=0] elapsed=4.1s

$ /usr/bin/python3 tools/export_pruned.py --config configs/ablations/hbcc_accuracy_small_pruning_mask.yaml --checkpoint runs_pruning_accuracy_cifar100/hbcc_accuracy_small_pruning_mask_cifar100/best.pth --output configs/generated_ablations/hbcc_accuracy_small_cifar100_pruned_export.yaml --threshold 0.90 --min-channels 16 --round-to 8 --override data.name=cifar100 --override model.num_classes=100 --override experiment.name=hbcc_accuracy_small_cifar100_pruned_export


{'experiment': {'name': 'hbcc_accuracy_small_cifar100_pruned_export'}, 'data': {'name': 'cifar100', 'root': 'data', 'download': True, 'batch_size': 128, 'val_batch_size': 256, 'test_batch_size': 256, 'val_size': 5000, 'split_seed': 42, 'workers': 2, 'augment': True, 'randaugment': {'enabled': True, 'num_ops': 2, 'magnitude': 9}, 'random_erasing': {'p': 0.25, 'scale': [0.02, 0.2], 'ratio': [0.3, 3.3]}}, 'model': {'name': 'hbcc', 'num_classes': 100, 'use_coord': True, 'embed_dims': [48, 80, 160, 224], 'depths': [2, 2, 3, 1], 'mlp_ratios': 3.0, 'heads': [2, 2, 4, 4], 'head_dim': [16, 16, 16, 16], 'proposals': [[2, 2], [2, 2], [2, 2], [2, 2]], 'folds': [[4, 4], [2, 2], [1, 1], [1, 1]], 'similarities': ['cosine', 'cosine', 'cosine', 'cosine'], 'stage_modes': ['hybrid', 'hybrid', 'cluster', 'cluster'], 'local_branches': ['lbpconv', 'dwconv', 'identity', 'identity'], 'local_ratios': [0.5, 0.5, 0.0, 0.0], 'channel_shuffle': [True, True, False, False], 'norm': 'bn', 'stem_stride': 2, 'drop_path

```text
test: 100%|████████████| 40/40 [00:01<00:00, 19.89it/s]
```

setup: dataset=cifar100 output=runs_pruned_accuracy_cifar100/hbcc_accuracy_small_cifar100_pruned_export device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.8s
setup: building model...
setup: model ready params=1558868 elapsed=3.1s
train: starting epochs=80 skip_test=False

                                                                                    

                                                      
{
  "epoch": 0,
  "lr": 0.0001999229036240723,
  "train_loss": 4.259155058792853,
  "train_acc1": 7.449697293447294,
  "train_time_s": 27.2084338489949,
  "val_loss": 3.7292983646392823,
  "val_acc1": 14.34,
  "val_time_s": 1.5897697859982145,
  "val_acc5": 37.28
}

                                                                                     

                                                      
{
  "epoch": 1,
  "lr": 0.0001996917333733128,
  "train_loss": 3.968

## 8. Benchmark Matrix

In [13]:
BENCHMARK_JOBS = [
    ("configs/baselines/resnet18_cifar.yaml",            "resnet18_cifar100",                "runs_teacher_cifar100/resnet18_cifar100/best.pth"),
    ("configs/baselines/mobilenet_v2_cifar.yaml",         "mobilenet_v2_cifar100",             "runs_baselines_cifar100/mobilenet_v2_cifar100/best.pth"),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml",  "shufflenet_v2_x1_0_cifar100",       "runs_baselines_cifar100/shufflenet_v2_x1_0_cifar100/best.pth"),
    ("configs/coc_cifar_baseline.yaml",                   "coc_cifar_baseline_cifar100",       COC_BASELINE_CHECKPOINT),
    ("configs/hbcc_current_reference.yaml",               "hbcc_current_reference_cifar100",   None),
    ("configs/hbcc_latency_tiny.yaml",                    "hbcc_latency_tiny_cifar100",        "runs_students_cifar100/hbcc_latency_tiny_cifar100/best.pth"),
    ("configs/hbcc_latency_small.yaml",                   "hbcc_latency_small_cifar100",       "runs_students_cifar100/hbcc_latency_small_cifar100/best.pth"),
    ("configs/hbcc_accuracy_small.yaml",                  "hbcc_accuracy_small_cifar100",      "runs_accuracy_cifar100/hbcc_accuracy_small_cifar100/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml",                 "hbcc_accuracy_medium_cifar100",     "runs_accuracy_cifar100/hbcc_accuracy_medium_cifar100/best.pth"),
    # KD variants
    ("configs/hbcc_latency_tiny.yaml",   "hbcc_latency_tiny_kd_cifar100",    "runs_kd_cifar100/hbcc_latency_tiny_kd_cifar100/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml","hbcc_accuracy_medium_kd_cifar100", "runs_kd_cifar100/hbcc_accuracy_medium_kd_cifar100/best.pth"),
    ("configs/hbcc_accuracy_small.yaml", "hbcc_accuracy_small_kd_cifar100",  "runs_kd_cifar100/hbcc_accuracy_small_kd_cifar100/best.pth"),
    ("configs/ablations/hbcc_tiny_hamming_late.yaml", "hbcc_tiny_hamming_late_cifar100", "runs_ablations_cifar100/hbcc_tiny_hamming_late_cifar100/best.pth"),
    ("configs/ablations/hbcc_tiny_lbpconv.yaml",      "hbcc_tiny_lbpconv_cifar100",      "runs_ablations_cifar100/hbcc_tiny_lbpconv_cifar100/best.pth"),
    (PRUNED_CONFIG, PRUNED_FINETUNE_EXPERIMENT, PRUNED_FINETUNE_CHECKPOINT),
]

if RUN_BENCHMARKS:
    for cfg, experiment_name, ckpt in BENCHMARK_JOBS:
        if not (ROOT / cfg).exists():
            print("Skip missing config:", cfg)
            continue
        benchmark(cfg, experiment_name, checkpoint=ckpt, debug=False, profile=True)



$ /usr/bin/python3 tools/benchmark.py --config configs/baselines/resnet18_cifar.yaml --output results/benchmark_cifar100 --batch-sizes 1 16 64 128 --warmup 30 --runs 100 --override data.name=cifar100 --override model.num_classes=100 --override experiment.name=resnet18_cifar100 --checkpoint runs_teacher_cifar100/resnet18_cifar100/best.pth --profile


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "resnet18_cifar",
  "config_id": "resnet18_cifar100",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 11220132,
  "params_trainable": 11220132,
  "model_size_mb": 42.83818054199219,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 3.1711267300124746,
  "throughput_b1": 533.2192443205636,
  "latency_ms_b16": 5.559594859951176,
  "throughput_b16": 2922.317952445833,
  "latency_ms_b64": 15.193510809986037,
  "throughput_b64": 4194.094268327932,
  "latency_ms_b128": 29.412990029959474,
  "throughput_b128": 429

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "mobilenet_v2_cifar",
  "config_id": "mobilenet_v2_cifar100",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 2351972,
  "params_trainable": 2351972,
  "model_size_mb": 9.102584838867188,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 4.729672870016657,
  "throughput_b1": 220.3957312902341,
  "latency_ms_b16": 4.599817829948734,
  "throughput_b16": 3597.39060211967,
  "latency_ms_b64": 6.735636399971554,
  "throughput_b64": 9530.512892284205,
  "latency_ms_b128": 13.415288210017025,
  "throughput_b128": 

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "shufflenet_v2_x1_0_cifar",
  "config_id": "shufflenet_v2_x1_0_cifar100",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 1356104,
  "params_trainable": 1356104,
  "model_size_mb": 5.2352752685546875,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 6.075276090050465,
  "throughput_b1": 170.21914475649993,
  "latency_ms_b16": 6.560572140006116,
  "throughput_b16": 2468.7453787792497,
  "latency_ms_b64": 6.8159327899775235,
  "throughput_b64": 9464.174198934377,
  "latency_ms_b128": 13.924300339931506,
  "t

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "coc_cifar_baseline_cifar100",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 2424704,
  "params_trainable": 2424704,
  "model_size_mb": 9.24951171875,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 10.75293479996617,
  "throughput_b1": 92.28298003304249,
  "latency_ms_b16": 11.03025777003495,
  "throughput_b16": 1463.4166717705905,
  "latency_ms_b64": 10.97631975004333,
  "throughput_b64": 5818.651658413968,
  "latency_ms_b128": 14.571537689989782,
  "throughput_b128": 8884.47300

...

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_accuracy_medium_cifar100",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 2863992,
  "params_trainable": 2859384,
  "model_size_mb": 10.962379455566406,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 16.77839849005977,
  "throughput_b1": 61.57087839734671,
  "latency_ms_b16": 16.20068641997932,
  "throughput_b16": 1013.6526127180285,
  "latency_ms_b64": 16.998613120013033,
  "throughput_b64": 3910.2904847295395,
  "latency_ms_b128": 22.68523031998484,
  "throughput_b128": 56

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_latency_tiny_kd_cifar100",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 462236,
  "params_trainable": 462236,
  "model_size_mb": 1.7746658325195312,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 6.387379570005578,
  "throughput_b1": 157.59129993647895,
  "latency_ms_b16": 6.609974559978582,
  "throughput_b16": 2497.8154574458845,
  "latency_ms_b64": 6.839784129988402,
  "throughput_b64": 9334.578245185778,
  "latency_ms_b128": 6.621729420003248,
  "throughput_b128": 19709

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_tiny_hamming_late_cifar100",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 462236,
  "params_trainable": 462236,
  "model_size_mb": 1.7746658325195312,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 5120,
  "latency_ms_b1": 6.682195109970053,
  "throughput_b1": 150.38683984379998,
  "latency_ms_b16": 6.700274850009009,
  "throughput_b16": 2417.4876816218734,
  "latency_ms_b64": 6.799314310046611,
  "throughput_b64": 9487.309866142223,
  "latency_ms_b128": 6.908393600024283,
  "throughput_b128": 

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_tiny_lbpconv_cifar100",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 477748,
  "params_trainable": 473716,
  "model_size_mb": 1.8368301391601562,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 6.318045410007471,
  "throughput_b1": 158.49764771221393,
  "latency_ms_b16": 6.403904340040754,
  "throughput_b16": 2499.948751054854,
  "latency_ms_b64": 6.436574609979289,
  "throughput_b64": 10159.455253477821,
  "latency_ms_b128": 6.579937040005461,
  "throughput_b128": 19926.27

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_accuracy_small_cifar100_pruned_export",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 1558868,
  "params_trainable": 1555412,
  "model_size_mb": 5.971611022949219,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 11.785441849933704,
  "throughput_b1": 82.50828697350512,
  "latency_ms_b16": 11.982470540024224,
  "throughput_b16": 1295.7485265537446,
  "latency_ms_b64": 12.123910559967044,
  "throughput_b64": 5294.639661061411,
  "latency_ms_b128": 17.409200740003143,
  "throug

## 9. Read Metrics And Build Summary Tables

In [14]:
CIFAR100_RUN_DIRS = [
    "runs_teacher_cifar100",
    "runs_baselines_cifar100",
    "runs_coc_baseline_cifar100",
    "runs_students_cifar100",
    "runs_accuracy_cifar100",
    "runs_proxy_cifar100",
    "runs_ablations_cifar100",
    "runs_kd_cifar100",
    "runs_pruning_accuracy_cifar100",
    "runs_pruned_accuracy_cifar100",
]


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def collect_train_metrics() -> pd.DataFrame:
    rows = []
    for run_root in CIFAR100_RUN_DIRS:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records = [r for r in records if "test_acc1" in r]
            test_record = next((r for r in reversed(test_records) if r.get("epoch") == best.get("epoch")), test_records[-1] if test_records else {})
            train_records = [r for r in records if "train_acc1" in r]
            last_train = train_records[-1] if train_records else {}
            rows.append({
                "run_dir": str(metrics_path.parent.relative_to(ROOT)),
                "experiment": metrics_path.parent.name,
                "best_epoch": best.get("epoch"),
                "best_val_acc1": best.get("val_acc1"),
                "best_val_acc5": best.get("val_acc5"),
                "best_val_loss": best.get("val_loss"),
                "test_acc1": test_record.get("test_acc1"),
                "test_acc5": test_record.get("test_acc5"),
                "test_loss": test_record.get("test_loss"),
                "last_epoch": last_train.get("epoch"),
                "last_train_acc1": last_train.get("train_acc1"),
            })
    return pd.DataFrame(rows).sort_values("best_val_acc1", ascending=False) if rows else pd.DataFrame()


train_df = collect_train_metrics()
train_df

,run_dir,experiment,best_epoch,best_val_acc1,best_val_acc5,best_val_loss,test_acc1,test_acc5,test_loss,last_epoch,last_train_acc1
4,runs_accuracy_cifar100/hbcc_accuracy_medium_ci...,hbcc_accuracy_medium_cifar100,283,75.68,92.62,1.070067,75.37,92.43,1.082475,299,57.120281
0,runs_teacher_cifar100/resnet18_cifar100,resnet18_cifar100,182,74.84,91.30,1.332058,74.23,90.32,1.336980,199,99.973291
5,runs_accuracy_cifar100/hbcc_accuracy_small_cif...,hbcc_accuracy_small_cifar100,250,73.94,92.40,1.135207,73.01,92.04,1.163371,299,50.563123
6,runs_pruning_accuracy_cifar100/hbcc_accuracy_s...,hbcc_accuracy_small_pruning_mask_cifar100,67,73.94,91.70,1.104001,73.25,91.14,1.138038,79,93.732194
2,runs_baselines_cifar100/shufflenet_v2_x1_0_cif...,shufflenet_v2_x1_0_cifar100,177,70.68,89.18,1.405489,69.58,88.53,1.416602,199,99.964387
1,runs_baselines_cifar100/mobilenet_v2_cifar100,mobilenet_v2_cifar100,180,67.08,87.00,1.524180,66.16,86.26,1.546912,199,99.942130
3,runs_coc_baseline_cifar100/coc_cifar_baseline_...,coc_cifar_baseline_cifar100,191,60.82,80.40,1.843270,61.19,80.66,1.819580,199,99.289975
7,runs_pruned_accuracy_cifar100/hbcc_accuracy_sm...,hbcc_accuracy_small_cifar100_pruned_export,69,58.36,85.34,1.560719,58.44,84.68,1.552573,79,57.563212


In [15]:
def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / BENCHMARK_OUTPUT).glob("*.json"):
        rec = json.loads(path.read_text(encoding="utf-8"))
        rec["benchmark_file"] = str(path.relative_to(ROOT))
        rows.append(rec)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


bench_df = collect_benchmarks()
cols = [
    "config_id",
    "model_name",
    "params_total",
    "macs",
    "bops",
    "latency_ms_b1",
    "throughput_b16",
    "throughput_b64",
    "throughput_b128",
    "peak_memory_mb",
]
bench_df[[c for c in cols if c in bench_df.columns]] if not bench_df.empty else bench_df

,config_id,model_name,params_total,macs,bops,latency_ms_b1,throughput_b16,throughput_b64,throughput_b128,peak_memory_mb
0,mobilenet_v2_cifar100,mobilenet_v2_cifar,2351972,None,0,4.729673,3597.390602,9530.512892,9553.562702,123.804688
1,hbcc_accuracy_medium_cifar100,hbcc,2863992,None,0,16.778398,1013.652613,3910.290485,5646.020958,145.653809
2,hbcc_latency_tiny_kd_cifar100,hbcc,462236,None,0,6.387380,2497.815457,9334.578245,19709.737104,44.445312
3,hbcc_latency_tiny_cifar100,hbcc,462236,None,0,6.325625,2356.070436,9966.076659,19793.317713,44.445312
4,hbcc_tiny_lbpconv_cifar100,hbcc,477748,None,0,6.318045,2499.948751,10159.455253,19926.274280,116.504883
5,hbcc_latency_small_cifar100,hbcc,920302,None,0,7.917342,2011.909610,8049.086944,16008.836858,54.199219
6,resnet18_cifar100,resnet18_cifar,11220132,None,0,3.171127,2922.317952,4194.094268,4296.383381,326.540527
7,hbcc_current_reference_cifar100,hbcc,887470,None,0,6.823200,2338.410466,7040.455114,7152.570705,174.059082
8,hbcc_tiny_hamming_late_cifar100,hbcc,462236,None,5120,6.682195,2417.487682,9487.309866,18790.263102,44.445312
9,coc_cifar_baseline_cifar100,hbcc,2424704,None,0,10.752935,1463.416672,5818.651658,8884.473010,71.908691


## 10. Pareto Summary

In [16]:
def attach_accuracy(bench: pd.DataFrame, train_metrics: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        return bench
    out = bench.copy()
    out["acc1"] = None
    out["acc5"] = None
    # Fall back to committed published results for models not trained this session.
    # This lets baselines-only and HBCC-only runs each produce a complete table.
    fallback_csv = ROOT / "results" / "cifar100_published_results.csv"
    fallback_acc1: dict = {}
    fallback_acc5: dict = {}
    if fallback_csv.exists():
        try:
            fb = pd.read_csv(fallback_csv)
            if "Model" in fb.columns and "Top-1 Acc (%)" in fb.columns:
                fallback_acc1 = dict(zip(fb["Model"], fb["Top-1 Acc (%)"]))
            if "Model" in fb.columns and "Top-5 Acc (%)" in fb.columns:
                fallback_acc5 = dict(zip(fb["Model"], fb["Top-5 Acc (%)"]))
        except Exception:
            pass
    if not train_metrics.empty:
        acc1_map = dict(zip(train_metrics["experiment"], train_metrics["test_acc1"].fillna(train_metrics["best_val_acc1"])))
        acc5_source = train_metrics["test_acc5"].fillna(train_metrics["best_val_acc5"]) if "test_acc5" in train_metrics else None
        acc5_map = dict(zip(train_metrics["experiment"], acc5_source)) if acc5_source is not None else {}
    else:
        acc1_map, acc5_map = {}, {}
    for idx, row in out.iterrows():
        name = row.get("config_id")
        out.at[idx, "acc1"] = acc1_map.get(name, fallback_acc1.get(name))
        out.at[idx, "acc5"] = acc5_map.get(name, fallback_acc5.get(name))
    return out


def pareto_frontier(df: pd.DataFrame) -> pd.DataFrame:
    required = ["acc1", "params_total", "latency_ms_b1", "peak_memory_mb"]
    if df.empty or any(c not in df.columns for c in required):
        return pd.DataFrame()
    valid = df.dropna(subset=required).copy()
    keep = []
    for i, row in valid.iterrows():
        dominated = False
        for j, other in valid.iterrows():
            if i == j:
                continue
            better_or_equal = (
                other["acc1"] >= row["acc1"]
                and other["params_total"] <= row["params_total"]
                and other["latency_ms_b1"] <= row["latency_ms_b1"]
                and other["peak_memory_mb"] <= row["peak_memory_mb"]
            )
            strictly_better = (
                other["acc1"] > row["acc1"]
                or other["params_total"] < row["params_total"]
                or other["latency_ms_b1"] < row["latency_ms_b1"]
                or other["peak_memory_mb"] < row["peak_memory_mb"]
            )
            if better_or_equal and strictly_better:
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return valid.loc[keep].sort_values(["acc1", "latency_ms_b1"], ascending=[False, True])


combined_df = attach_accuracy(bench_df, train_df)
frontier_df = pareto_frontier(combined_df)

out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
if not combined_df.empty:
    combined_df.to_csv(out_dir / "cifar100_combined_training_benchmark.csv", index=False)
if not frontier_df.empty:
    frontier_df.to_csv(out_dir / "cifar100_pareto_frontier.csv", index=False)

frontier_df[[c for c in ["config_id", "acc1", "acc5", "params_total", "macs", "bops", "latency_ms_b1", "peak_memory_mb"] if c in frontier_df.columns]]

,config_id,acc1,acc5,params_total,macs,bops,latency_ms_b1,peak_memory_mb
1,hbcc_accuracy_medium_cifar100,75.37,92.43,2863992,None,0,16.778398,145.653809
6,resnet18_cifar100,74.23,90.32,11220132,None,0,3.171127,326.540527
10,hbcc_accuracy_small_cifar100,73.01,92.04,1558868,None,0,11.724616,109.668945
11,shufflenet_v2_x1_0_cifar100,69.58,88.53,1356104,None,0,6.075276,95.184082
0,mobilenet_v2_cifar100,66.16,86.26,2351972,None,0,4.729673,123.804688
9,coc_cifar_baseline_cifar100,61.19,80.66,2424704,None,0,10.752935,71.908691


## 11. Published Results

Final headline table for reporting: Top-1/Top-5 accuracy alongside params, MACs, BOPs, batch-1 latency, and peak memory for every model that has both a training run and a benchmark record. Sorted by Top-1 accuracy, descending. This is the table to cite in the paper.


In [17]:
def build_publication_table(df: pd.DataFrame) -> pd.DataFrame:
    """Rename/round combined training+benchmark records into a paper-ready table."""
    if df.empty:
        return df
    rename = {
        "config_id": "Model",
        "acc1": "Top-1 Acc (%)",
        "acc5": "Top-5 Acc (%)",
        "params_total": "Params",
        "macs": "MACs",
        "bops": "BOPs",
        "latency_ms_b1": "Latency b1 (ms)",
        "peak_memory_mb": "Peak Memory (MB)",
    }
    cols = [c for c in rename if c in df.columns]
    table = df[cols].rename(columns=rename).copy()
    if "Top-1 Acc (%)" in table.columns:
        table = table.sort_values("Top-1 Acc (%)", ascending=False)
    return table.reset_index(drop=True)


publication_df = build_publication_table(combined_df)
publication_df.to_csv(out_dir / "cifar100_published_results.csv", index=False)
publication_df


,Model,Top-1 Acc (%),Top-5 Acc (%),Params,MACs,BOPs,Latency b1 (ms),Peak Memory (MB)
0,hbcc_accuracy_medium_cifar100,75.37,92.43,2863992,None,0,16.778398,145.653809
1,resnet18_cifar100,74.23,90.32,11220132,None,0,3.171127,326.540527
2,hbcc_accuracy_small_cifar100,73.01,92.04,1558868,None,0,11.724616,109.668945
3,shufflenet_v2_x1_0_cifar100,69.58,88.53,1356104,None,0,6.075276,95.184082
4,mobilenet_v2_cifar100,66.16,86.26,2351972,None,0,4.729673,123.804688
5,coc_cifar_baseline_cifar100,61.19,80.66,2424704,None,0,10.752935,71.908691
6,hbcc_accuracy_small_cifar100_pruned_export,58.44,84.68,1558868,None,0,11.785442,109.668945
7,hbcc_latency_tiny_kd_cifar100,None,None,462236,None,0,6.387380,44.445312
8,hbcc_latency_tiny_cifar100,None,None,462236,None,0,6.325625,44.445312
9,hbcc_tiny_lbpconv_cifar100,None,None,477748,None,0,6.318045,116.504883


In [18]:
if RUN_SUMMARY_REPORT:
    run_py([
        "tools/cifar100_report.py",
        "--runs",
        *CIFAR100_RUN_DIRS,
        "--benchmarks",
        BENCHMARK_OUTPUT,
        "--output",
        "results/cifar100_summary.csv",
    ])

if RUN_PARETO_REPORT:
    run_py(["tools/pareto_report.py", BENCHMARK_OUTPUT, "--output", "results/cifar100_pareto.md"])


$ /usr/bin/python3 tools/cifar100_report.py --runs runs_teacher_cifar100 runs_baselines_cifar100 runs_coc_baseline_cifar100 runs_students_cifar100 runs_accuracy_cifar100 runs_proxy_cifar100 runs_ablations_cifar100 runs_kd_cifar100 runs_pruning_accuracy_cifar100 runs_pruned_accuracy_cifar100 --benchmarks results/benchmark_cifar100 --output results/cifar100_summary.csv


wrote results/cifar100_summary.csv rows=8

[exit=0] elapsed=0.1s

$ /usr/bin/python3 tools/pareto_report.py results/benchmark_cifar100 --output results/cifar100_pareto.md


results/cifar100_pareto.md

[exit=0] elapsed=0.6s


## Recommended Execution Order

1. Run environment checks and smoke.
2. Enable `RUN_TEACHER` and train ResNet-18 on CIFAR-100.
3. Enable `RUN_COC_BASELINE` and train the CoC baseline with CIFAR-100 overrides.
4. Enable `RUN_STUDENT_CE` and train HBCC-Accuracy Small/Medium (hbcc_latency_tiny/small are intentionally skipped here to stay inside Kaggle's 12h session cap, matching the executed CIFAR-10 run).
5. Enable `RUN_PROXY_SEARCH`; use the table to choose top candidates.
6. Enable `RUN_ABLATIONS` for Hamming/LBPConv.
7. Enable `RUN_KD` after the CIFAR-100 teacher checkpoint exists.
8. Enable `RUN_PRUNING` after `runs_accuracy_cifar100/hbcc_accuracy_small_cifar100/best.pth` exists.
9. Enable `RUN_BENCHMARKS` and build final tables.

Do not treat proxy accuracy or untrained benchmark records as final scientific evidence.